# Random Forest V1

For this first experiment, I tested a Random Forest on both datasets. I used 5 trees and the same training and validation samples as my earlier Decision Tree experiments.

First, I imported the libraries and functions I'll need. The forest follows Lab 5 cells 40 and 43 and uses my existing Decision Tree.

In [ ]:
from pathlib import Path
from time import perf_counter
from collections import Counter
import sys
import platform
import numpy as np
import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
from src.data_loader import load_task, split_train_val
from src.models.random_forest import RandomForest, bootstrap_sample
from src.models.decision_tree import predict_one

data_dir = project_root.parent / "data"
result_dir = project_root / "results" / "random_forest"
summary_rows = []
print(f"Python {platform.python_version()}, NumPy {np.__version__}, pandas {pd.__version__}")
print(f"Data folder: {data_dir}")


Python 3.14.3, NumPy 2.4.3, pandas 3.0.1
Data folder: /Users/jidapalaopikanon/Desktop/COMP472/data


I checked the bootstrap samples, voting, tree depth and OOB calculation on a small artificial dataset first.

In [ ]:
tiny_X = [[float(i), float(i % 3)] for i in range(12)]
tiny_y = [i % 2 for i in range(12)]
boot_X, boot_y, oob = bootstrap_sample(tiny_X, tiny_y, seed=42)
sampled = [int(row[0]) for row in boot_X]
assert len(boot_X) == len(boot_y) == 12
assert len(set(sampled)) < 12
assert boot_y == [tiny_y[i] for i in sampled]
assert oob == sorted(set(range(12)) - set(sampled))
assert (boot_X, boot_y, oob) == bootstrap_sample(tiny_X, tiny_y, seed=42)

def tree_depth(node):
    if node.is_leaf():
        return 0
    return 1 + max(tree_depth(node.left), tree_depth(node.right))

tiny = RandomForest(n_trees=5, max_depth=2, seed=42)
assert tiny.fit(tiny_X, tiny_y) is tiny
assert len(tiny.trees) == 5
assert all(tree_depth(t.root) <= 2 for t in tiny.trees)
expected_votes = {i: [] for i in range(12)}
for t, tree in enumerate(tiny.trees):
    _, _, excluded = bootstrap_sample(tiny_X, tiny_y, seed=42 + t)
    for i in excluded:
        expected_votes[i].append(predict_one(tree.root, tiny_X[i]))
assert tiny._oob_preds == expected_votes
votes = [[predict_one(t.root, x) for t in tiny.trees] for x in tiny_X]
expected = [Counter(v).most_common(1)[0][0] for v in votes]
assert tiny.predict(tiny_X) == expected
assert len(expected) == 12 and set(expected) <= {0, 1}
covered = [i for i, v in expected_votes.items() if v]
correct = sum(Counter(expected_votes[i]).most_common(1)[0][0] == tiny_y[i] for i in covered)
assert tiny.oob_score(tiny_y) == correct / len(covered)
repeat = RandomForest(n_trees=5, max_depth=2, seed=42).fit(tiny_X, tiny_y)
assert repeat.predict(tiny_X) == expected
assert repeat._oob_preds == tiny._oob_preds
assert tiny.fit(tiny_X, tiny_y).predict(tiny_X) == expected
assert len(tiny.trees) == 5 and tiny._oob_preds == expected_votes
probe = RandomForest()
probe._oob_preds = {0: [], 1: [1, 0], 2: [0]}
assert probe.oob_score([1, 1, 1]) == 0.5
probe._oob_preds = {0: []}
assert probe.oob_score([1]) == 0.0
assert tiny._get_k(500) == 22 and tiny._get_k(768) == 27
print("Small-data checks passed, including deterministic voting and OOB-only scoring.")


Small-data checks passed, including deterministic voting and OOB-only scoring.


I started by testing the model on the image dataset.

In [ ]:
X, y, _, _ = load_task("image", data_dir=data_dir)
X_train_full, X_val_full, y_train_full, y_val_full = split_train_val(X, y, val_fraction=0.2, seed=42)
image_X_train = X_train_full[:200]
image_y_train = y_train_full[:200]
image_X_val = X_val_full[:200]
image_y_val = y_val_full[:200]
assert image_X_train.shape == image_X_val.shape == (200, 500)
dt_validation = pd.read_csv(project_root / "results/decision_tree/decision_tree_v2_image_validation.csv")
assert dt_validation["validation_index"].tolist() == list(range(200))
assert dt_validation["actual_label"].tolist() == image_y_val.tolist()

image_clf = RandomForest(n_trees=5, max_depth=2, min_samples=1, n_features="sqrt", seed=42)
start = perf_counter()
image_clf.fit(image_X_train, image_y_train)
image_fit_time = perf_counter() - start
image_training_accuracy = image_clf.score(image_X_train, image_y_train)
start = perf_counter()
image_predictions = image_clf.predict(image_X_val)
image_prediction_time = perf_counter() - start
image_validation_accuracy = sum(p == t for p, t in zip(image_predictions, image_y_val)) / 200
image_oob_accuracy = image_clf.oob_score(image_y_train)
image_oob_covered = sum(bool(v) for v in image_clf._oob_preds.values())
assert len(image_clf.trees) == 5
assert all(t.criterion == "entropy" and tree_depth(t.root) <= 2 for t in image_clf.trees)
assert len(image_predictions) == 200
assert all(0 <= p <= 49 for p in image_predictions)
summary_rows.append({
    "dataset": "image", "version": "V1", "n_trees": 5,
    "max_depth": 2, "min_samples": 1, "n_features": "sqrt",
    "criterion": "entropy", "normalized": False,
    "training_samples": 200, "validation_samples": 200,
    "training_accuracy": image_training_accuracy,
    "validation_accuracy": image_validation_accuracy,
    "oob_accuracy": image_oob_accuracy,
    "fit_time_seconds": image_fit_time,
    "prediction_time_seconds": image_prediction_time,
})
print(f"Image: {len(image_clf.trees)} trees, {image_clf._get_k(500)} features per split")
print(f"Training={image_training_accuracy:.6%}, validation={image_validation_accuracy:.6%}")
print(f"OOB={image_oob_accuracy:.6%} on {image_oob_covered}/200 training examples")
print(f"Fit={image_fit_time:.9f} s, validation prediction={image_prediction_time:.9f} s")


Image: 5 trees, 22 features per split
Training=9.000000%, validation=2.500000%
OOB=2.247191% on 178/200 training examples
Fit=0.120858084 s, validation prediction=0.000404625 s


Then I repeated the same test with the sentiment dataset.

In [4]:
X, y, _, _ = load_task("sentiment", data_dir=data_dir)
X_train_full, X_val_full, y_train_full, y_val_full = split_train_val(X, y, val_fraction=0.2, seed=42)
sentiment_X_train = X_train_full[:200]
sentiment_y_train = y_train_full[:200]
sentiment_X_val = X_val_full[:200]
sentiment_y_val = y_val_full[:200]
assert sentiment_X_train.shape == sentiment_X_val.shape == (200, 768)
dt_validation = pd.read_csv(project_root / "results/decision_tree/decision_tree_v2_sentiment_validation.csv")
assert dt_validation["validation_index"].tolist() == list(range(200))
assert dt_validation["actual_label"].tolist() == sentiment_y_val.tolist()

sentiment_clf = RandomForest(n_trees=5, max_depth=3, min_samples=1, n_features="sqrt", seed=42)
start = perf_counter()
sentiment_clf.fit(sentiment_X_train, sentiment_y_train)
sentiment_fit_time = perf_counter() - start
sentiment_training_accuracy = sentiment_clf.score(sentiment_X_train, sentiment_y_train)
start = perf_counter()
sentiment_predictions = sentiment_clf.predict(sentiment_X_val)
sentiment_prediction_time = perf_counter() - start
sentiment_validation_accuracy = sum(p == t for p, t in zip(sentiment_predictions, sentiment_y_val)) / 200
sentiment_oob_accuracy = sentiment_clf.oob_score(sentiment_y_train)
sentiment_oob_covered = sum(bool(v) for v in sentiment_clf._oob_preds.values())
assert len(sentiment_clf.trees) == 5
assert all(t.criterion == "entropy" and tree_depth(t.root) <= 3 for t in sentiment_clf.trees)
assert len(sentiment_predictions) == 200
assert all(0 <= p <= 2 for p in sentiment_predictions)
summary_rows.append({
    "dataset": "sentiment", "version": "V1", "n_trees": 5,
    "max_depth": 3, "min_samples": 1, "n_features": "sqrt",
    "criterion": "entropy", "normalized": False,
    "training_samples": 200, "validation_samples": 200,
    "training_accuracy": sentiment_training_accuracy,
    "validation_accuracy": sentiment_validation_accuracy,
    "oob_accuracy": sentiment_oob_accuracy,
    "fit_time_seconds": sentiment_fit_time,
    "prediction_time_seconds": sentiment_prediction_time,
})
print(f"Sentiment: {len(sentiment_clf.trees)} trees, {sentiment_clf._get_k(768)} features per split")
print(f"Training={sentiment_training_accuracy:.6%}, validation={sentiment_validation_accuracy:.6%}")
print(f"OOB={sentiment_oob_accuracy:.6%} on {sentiment_oob_covered}/200 training examples")
print(f"Fit={sentiment_fit_time:.9f} s, validation prediction={sentiment_prediction_time:.9f} s")


Sentiment: 5 trees, 27 features per split
Training=69.500000%, validation=38.500000%
OOB=43.820225% on 178/200 training examples
Fit=0.895715584 s, validation prediction=0.000508917 s


Finally, I saved the predictions and compared the accuracies with my Decision Tree results. OOB accuracy uses only training examples with OOB votes; validation accuracy uses the separate held-out examples.

In [5]:
result_dir.mkdir(parents=True, exist_ok=True)
for task, actual, predicted in [
    ("image", image_y_val, image_predictions),
    ("sentiment", sentiment_y_val, sentiment_predictions),
]:
    path = result_dir / f"random_forest_v1_{task}_validation.csv"
    pd.DataFrame({"validation_index": range(200), "actual_label": actual,
                  "predicted_label": predicted}).to_csv(path, index=False)
    saved = pd.read_csv(path)
    assert saved.shape == (200, 3) and not saved.isna().any().any()
    assert saved["validation_index"].tolist() == list(range(200))
    assert saved["actual_label"].tolist() == actual.tolist()
    assert saved["predicted_label"].tolist() == list(predicted)
    row = next(r for r in summary_rows if r["dataset"] == task)
    assert sum(saved.actual_label == saved.predicted_label) / 200 == row["validation_accuracy"]

summary = pd.DataFrame(summary_rows)
summary.to_csv(result_dir / "random_forest_v1_summary.csv", index=False)
assert len(pd.read_csv(result_dir / "random_forest_v1_summary.csv")) == 2
print(summary.to_string(index=False))

dt_summary = pd.read_csv(project_root / "results/decision_tree/decision_tree_v2_summary.csv")
for row in summary_rows:
    dt = dt_summary.loc[(dt_summary.dataset == row["dataset"]) & (dt_summary.max_depth == row["max_depth"])].iloc[0]
    assert dt.training_samples == dt.validation_samples == 200
    gap = 100 * (row["validation_accuracy"] - dt.validation_accuracy)
    print(f"{row['dataset']}: RF {row['validation_accuracy']:.2%}, DT {dt.validation_accuracy:.2%}, difference {gap:+.2f} percentage points")

tracker_path = result_dir / "kaggle_scores.csv"
columns = ["version", "image_validation_accuracy", "sentiment_validation_accuracy",
           "kaggle_public_score", "submitted_to_kaggle", "submission_filename", "notes"]
tracker = pd.read_csv(tracker_path, dtype=str, keep_default_na=False) if tracker_path.exists() else pd.DataFrame(columns=columns)
assert tracker["version"].is_unique
v1_row = {"version": "V1", "image_validation_accuracy": image_validation_accuracy,
          "sentiment_validation_accuracy": sentiment_validation_accuracy,
          "kaggle_public_score": "", "submitted_to_kaggle": False, "submission_filename": "",
          "notes": "5 trees; image depth=2, sentiment depth=3; sqrt features per split; min_samples=1; entropy; seed=42; original features; 200 training/200 validation per task after stratified split. No Kaggle candidate."}
matching = tracker["version"] == "V1"
if matching.any():
    for column in ("image_validation_accuracy", "sentiment_validation_accuracy", "notes"):
        tracker.loc[matching, column] = v1_row[column]
else:
    tracker = pd.concat([tracker, pd.DataFrame([v1_row])], ignore_index=True)
assert tracker["version"].is_unique
tracker.to_csv(tracker_path, index=False)
print(tracker.to_string(index=False))


  dataset version  n_trees  max_depth  min_samples n_features criterion  normalized  training_samples  validation_samples  training_accuracy  validation_accuracy  oob_accuracy  fit_time_seconds  prediction_time_seconds
    image      V1        5          2            1       sqrt   entropy       False               200                 200              0.090                0.025      0.022472          0.120858                 0.000405
sentiment      V1        5          3            1       sqrt   entropy       False               200                 200              0.695                0.385      0.438202          0.895716                 0.000509
image: RF 2.50%, DT 4.50%, difference -2.00 percentage points
sentiment: RF 38.50%, DT 45.00%, difference -6.50 percentage points
version image_validation_accuracy sentiment_validation_accuracy kaggle_public_score submitted_to_kaggle submission_filename                                                                                          

With 5 trees, my image validation accuracy was 2.50%, compared with 4.50% for the same-depth Decision Tree. My sentiment validation accuracy was 38.50%, compared with 45.00% for the Decision Tree. These comparisons use the same held-out samples, but each forest tree learns from a bootstrap sample and random split features.